# Augur Tutorial: Fisher Matrix Forecasting for DESC 3x2pt Analyses

**Augur** is a DESC forecasting and inference-validation tool. It acts as a wrapper around
[Firecrown](https://github.com/LSSTDESC/firecrown) and [pyccl](https://github.com/LSSTDESC/CCL)
to:

1. **Generate** a synthetic fiducial data vector and covariance matrix (stored in a SACC file).
2. **Analyze** — compute a Fisher matrix forecast by taking numerical derivatives of the likelihood.
3. **Postprocess** — produce triangle plots, pair-parameter plots, and a Figure-of-Merit LaTeX table.

Everything is driven by a single **YAML configuration file**. This notebook walks through each step
using the `srd_y1_3x2.yml` example (an LSST Year-1 3×2pt forecast).

> **Prerequisites**
> - A conda environment with `firecrown ≥ 1.14`, `pyccl ≥ 3.3.1`, and `augur` installed.
> - `augur` installed in the active conda environment (e.g. `pip install -e /path/to/augur`).
> - Run `conda install -c conda-forge isitgr` if you plan to use modified-gravity configs.

---
## 1. Setup

Set `TUTORIAL_DIR` to the notebook's own directory. Config files use
`{{ env['TUTORIAL_DIR'] }}/...`) to resolve data-file paths.

In [ ]:
import os
import pathlib

# Set TUTORIAL_DIR to the directory containing this notebook
TUTORIAL_DIR = str(pathlib.Path(".").resolve())
os.environ["TUTORIAL_DIR"] = TUTORIAL_DIR
print(f"TUTORIAL_DIR = {TUTORIAL_DIR}")

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
import sacc

import augur
from augur.generate import generate
from augur.analyze import Analyze
from augur.postprocess import postprocess, draw_fisher_ellipses, get_FoM_all

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Path to the example config we will use throughout this tutorial
CONFIG_PATH = os.path.join(TUTORIAL_DIR, "srd_y1_3x2.yml")
print(f"Using config: {CONFIG_PATH}")

---
## 2. The Configuration File

Augur is entirely driven by a YAML config. Let's look at the key sections.

In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    raw = f.read()

# Show the config (Jinja2 templates are rendered later by augur.parse)
print(raw)

In [ ]:
# Parse the config: Jinja2 templates are resolved and numpy expressions evaluated
parsed_config = augur.parse(CONFIG_PATH)

print("Top-level sections:", list(parsed_config.keys()))
print("\nFiducial cosmology:")
for k, v in parsed_config["cosmo"].items():
    if k != "extra_parameters":
        print(f"  {k}: {v}")
print("\nVaried parameters (Fisher):", parsed_config["fisher"]["var_pars"])

---
## 3. Generating the Fiducial Data Vector

`generate()` builds the Firecrown likelihood at the fiducial cosmology and writes a
SACC file containing:
- The fiducial theory $C_\ell$ data vector.
- The covariance matrix.
- Tracer metadata (N(z) bins).

The function returns the Firecrown `likelihood`, the SACC object `S`, `tools`
(pyccl-backed modeling tools), and `req_params` (fiducial parameter map).

In [ ]:
from copy import deepcopy

likelihood, S, tools, req_params = generate(
    deepcopy(parsed_config), return_all_outputs=True
)
print("Data vector length:", len(likelihood.get_data_vector()))
print("SACC data types:", [str(dt) for dt in set(d.data_type for d in S.data)])

### 3.1 Inspect the SACC file

The SACC file bundles all two-point observables. Let's visualise the three 3x2pt
probe types: galaxy clustering ($C_\ell^{gg}$), cosmic shear ($C_\ell^{\gamma\gamma}$),
and galaxy–galaxy lensing ($C_\ell^{g\gamma}$).

In [ ]:
sacc_path = parsed_config["fiducial_sacc_path"]
S_disk = sacc.Sacc.load_fits(sacc_path)

probe_labels = {
    "cl_00": ("galaxy_density_cl",  "gg",  "C"),
    "cl_ee": ("galaxy_shear_cl_ee", "γγ",  "C"),
    "cl_e":  ("galaxy_shearDensity_cl_e", "gγ", "C"),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (key, (dtype, label, _)) in zip(axes, probe_labels.items()):
    tracers = S_disk.get_tracer_combinations(dtype)
    for t1, t2 in tracers[:4]:   # show first 4 combinations to keep it readable
        ell, cl = S_disk.get_ell_cl(dtype, t1, t2)
        ax.plot(ell, ell * (ell + 1) * cl / (2 * np.pi), label=f"{t1}×{t2}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$\ell(\ell+1)C_\ell / 2\pi$")
    ax.set_title(f"${label}$")
    ax.legend(fontsize=7, ncol=2)

plt.suptitle("Fiducial 3×2pt Data Vector", fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Inspect the N(z) distributions

The SACC file also carries the per-bin redshift distributions for source and lens tracers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, prefix, title in zip(axes, ["src", "lens"], ["Source (shear)", "Lens (counts)"]):
    tracers = [t for t in S_disk.tracers if t.startswith(prefix)]
    for name in sorted(tracers):
        tr = S_disk.get_tracer(name)
        ax.plot(tr.z, tr.nz, label=name)
    ax.set_xlabel("z")
    ax.set_ylabel("N(z)")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle("Redshift Distributions", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Fisher Matrix Analysis

`Analyze` takes the likelihood and tools produced by `generate` and computes
the Fisher information matrix by numerical differentiation:

$$F_{ij} = \sum_{\alpha\beta} \frac{\partial \mu_\alpha}{\partial \theta_i}
C^{-1}_{\alpha\beta} \frac{\partial \mu_\beta}{\partial \theta_j}$$

The step size, derivative method (5-point stencil, numdifftools, derivkit),
parameter transforms ($\Omega_c \to \Omega_m$, $\sigma_8 \to S_8$), and
Gaussian priors are all configurable in the YAML `fisher:` section.

In [ ]:
analysis = Analyze(
    parsed_config,
    likelihood=likelihood,
    tools=tools,
    req_params=req_params,
)
analysis.get_fisher_matrix()

fisher = analysis.Fij
var_pars = analysis.var_pars
print("Varied parameters:", var_pars)
print("Fisher matrix shape:", fisher.shape)

### 4.1 Visualise the Fisher matrix

A well-conditioned Fisher matrix should be positive-definite with strong
diagonal entries (high information). Off-diagonal correlations reveal
parameter degeneracies.

In [ ]:
# Normalise for display: correlation matrix  C_ij = F_ij / sqrt(F_ii * F_jj)
diag = np.sqrt(np.diag(fisher))
corr = fisher / np.outer(diag, diag)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(len(var_pars)))
ax.set_yticks(range(len(var_pars)))
ax.set_xticklabels(var_pars, rotation=90, fontsize=8)
ax.set_yticklabels(var_pars, fontsize=8)
plt.colorbar(im, ax=ax, label="Correlation coefficient")
ax.set_title("Fisher Information Correlation Matrix")
plt.tight_layout()
plt.show()

### 4.2 Marginalised 1D constraints

Invert the Fisher matrix to get the parameter covariance. The marginalised
$1\sigma$ uncertainty on parameter $i$ is $\sqrt{(F^{-1})_{ii}}$.

In [ ]:
inv_fisher = np.linalg.inv(fisher)
sigmas = np.sqrt(np.diag(inv_fisher))

# Show only the cosmological parameters (first 7)
cosmo_pars = var_pars[:7]
cosmo_sigmas = sigmas[:7]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cosmo_pars, cosmo_sigmas, color="steelblue")
ax.set_xlabel(r"Marginalised $1\sigma$ uncertainty")
ax.set_title("Forecast Cosmological Parameter Constraints (Y1 3×2pt)")
for i, (par, sig) in enumerate(zip(cosmo_pars, cosmo_sigmas)):
    ax.text(sig * 1.02, i, f"{sig:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Fisher Bias (Systematic Shifts)

`get_fisher_bias()` estimates how a systematic parameter shift (e.g. a miscentred
photo-z or a biased galaxy bias) propagates into a bias on cosmological parameters.

The bias vector is:

$$\delta\theta_i = -F^{-1}_{ij} \sum_\alpha \frac{\partial\mu_\alpha}{\partial\theta_j}
C^{-1}_{\alpha\beta}\, \Delta\mu_\beta$$

where $\Delta\mu$ is the shift in the theory vector when the biased parameters
are adopted.

In [ ]:
# The bias_params to use are taken from the config's fisher_bias section
print("Bias scenario from config:")
for k, v in parsed_config["fisher"]["fisher_bias"]["bias_params"].items():
    print(f"  {k}: {v}")

analysis.get_fisher_bias()
bias = analysis.bi

fig, ax = plt.subplots(figsize=(8, 4))
# Normalise bias by the 1-sigma forecast
bias_in_sigma = bias / sigmas
ax.barh(var_pars, bias_in_sigma, color=["tomato" if abs(b) > 0.5 else "steelblue"
                                         for b in bias_in_sigma])
ax.axvline(0, color="k", linewidth=0.8)
ax.axvline( 0.5, color="gray", linewidth=0.8, linestyle="--", label="0.5σ threshold")
ax.axvline(-0.5, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel(r"Bias / $\sigma_{\rm marg}$")
ax.set_title("Systematic Bias in Units of Forecast $1\\sigma$")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Postprocessing: Ellipse Plots & Figure of Merit

The `postprocess` module produces:
- A **triangle plot** of marginalised 2D confidence ellipses.
- Individual **pair plots** for selected parameter pairs.
- A **FoM LaTeX table** for $w_0$–$w_a$.

Here we call the helper functions directly for more flexibility.

### 6.1 Dark energy $w_0$–$w_a$ ellipses

In [ ]:
keys = np.array(var_pars)
iw  = np.where(keys == "w0")[0][0]
iwa = np.where(keys == "wa")[0][0]

# Extract 2x2 sub-covariance for w0, wa
inv_sub = np.array([[inv_fisher[iw, iw],  inv_fisher[iw, iwa]],
                    [inv_fisher[iwa, iw], inv_fisher[iwa, iwa]]])

fig, ax = plt.subplots(figsize=(5, 5))
colors = ["steelblue", "tomato"]
cls    = [0.68, 0.95]
for cl, color in zip(cls, colors):
    draw_fisher_ellipses(ax, inv_sub, facecolors="none", edgecolors=color,
                         linestyles="-", linewidth=2,
                         mu=(-1.0, 0.0), CL=cl, alpha=1.0)
    ax.plot([], [], color=color, label=f"{int(cl*100)}% CL")

ax.set_xlim(-1.5, -0.5)
ax.set_ylim(-1.5,  1.5)
ax.axvline(-1, color="gray", linestyle="--", linewidth=0.8)
ax.axhline( 0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel(r"$w_0$")
ax.set_ylabel(r"$w_a$")
ax.set_title(r"Dark Energy Constraints: $w_0$–$w_a$ (Y1 3×2pt)")
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Figure of Merit

The DETF Figure of Merit (FoM) measures the constraining power on dark energy.
Two definitions are commonly used:

$$\text{FoM}_1 = \frac{1}{\sqrt{\sigma^2(w_0)\,\sigma^2(w_a) - \sigma^2(w_0 w_a)}}
\qquad
\text{FoM}_2 = \sqrt{\det F_{w_0 w_a}}$$

In [ ]:
for cl in [0.68, 0.95]:
    fom1, fom2 = get_FoM_all(fisher, iw, iwa, cl)
    sig_w0 = np.sqrt(inv_fisher[iw, iw])
    sig_wa = np.sqrt(inv_fisher[iwa, iwa])
    print(f"CL = {cl:.0%}")
    print(f"  σ(w0) = {sig_w0:.4f},  σ(wa) = {sig_wa:.4f}")
    print(f"  FoM₁  = {fom1:.1f}")
    print(f"  FoM₂  = {fom2:.1f}")

### 6.3 Running postprocess end-to-end

When using the CLI (`augur config.yml`) or calling `postprocess(config)` directly,
all plots are written to disk. The config's `postprocess:` section controls output
paths, colour scheme, and which pairs to plot.

In [ ]:
# postprocess reads the saved fisher.dat and fiducials.dat files.
# Make sure the output/ directory exists and the files were saved by the Analyze step.
os.makedirs(os.path.join(TUTORIAL_DIR, "output"), exist_ok=True)

# The postprocess section in srd_y1_3x2.yml drives all outputs
postprocess(augur.parse(CONFIG_PATH))
print("Plots written to", os.path.join(TUTORIAL_DIR, "output"))

---
## 7. Minimal 1-bin 3×2pt with TJPCov Covariance

The configs used so far have 5 tomographic bins each and use a pre-computed SRD
covariance matrix.  Here we build the simplest possible 3×2pt data vector —
**one source bin and one lens bin** — and let **TJPCov** compute a theoretical
Gaussian covariance on-the-fly.

### 7.1 What is TJPCov?

[TJPCov](https://github.com/LSSTDESC/TJPCov) (`tjpcov`) is a DESC library that
computes covariance matrices for two-point statistics.  Augur uses its
`FourierGaussianFsky` estimator, which implements the mode-counting Gaussian
approximation:

$$\text{Cov}\!\left[C_\ell^{AB}, C_{\ell'}^{CD}\right] =
\frac{\delta_{\ell\ell'}}{(2\ell+1)\,f_\text{sky}\,\Delta\ell}
\left[\tilde{C}_\ell^{AC}\tilde{C}_\ell^{BD} +
      \tilde{C}_\ell^{AD}\tilde{C}_\ell^{BC}\right]$$

where $\tilde{C}_\ell$ includes the noise power $N_\ell$ for auto-spectra.
TJPCov takes the CCL tracer objects directly from the firecrown likelihood,
so it is always consistent with the theory vector.

### 7.2 The 1-bin config

The key difference from the SRD config is:

```yaml
sources:
    nbins: 1

lenses:
    nbins: 1

cov_options:
    cov_type: tjpcov
    fsky: 0.3
    IA: null
    binning_info:
        ell_edges: np.geomspace(20, 2000, 11, endpoint=True)
```

> **Important**: `cov_options.binning_info.ell_edges` must be **identical** to the
> `ell_edges` used in each `statistics` block — TJPCov validates this.


In [ ]:
import os
os.makedirs(os.path.join(TUTORIAL_DIR, "output"), exist_ok=True)

# --- Build config paths ---
config_gaus_path   = os.path.join(TUTORIAL_DIR, "1bin_3x2pt_gaus.yml")
config_tjpcov_path = os.path.join(TUTORIAL_DIR, "1bin_3x2pt_tjpcov.yml")

# --- Generate the fiducial data vector with simple internal Gaussian covariance ---
config_gaus = augur.parse(config_gaus_path)
lk1, S1, tools1, req1 = generate(deepcopy(config_gaus), return_all_outputs=True)

print(f"Data vector length : {len(lk1.get_data_vector())} bins")
print(f"Covariance shape   : {S1.covariance.covmat.shape}")
print(f"Probes             : {sorted(set(d.data_type for d in S1.data))}")


### 7.3 Generate with TJPCov covariance

Now run `generate()` a second time with the TJPCov config.  The CCL tracers built
by firecrown are passed directly to TJPCov, which computes each covariance block
from the fiducial power spectra and the per-tracer noise levels.


In [ ]:
config_tjpcov = augur.parse(config_tjpcov_path)
lk2, S2, tools2, req2 = generate(deepcopy(config_tjpcov), return_all_outputs=True)

print(f"TJPCov covariance shape: {S2.covariance.covmat.shape}")

# --- Compare diagonal of gaus_internal vs TJPCov ---
import numpy as np

diag_gaus   = np.diag(S2.covariance.covmat)   # tjpcov keeps ignore_scale_cuts=True → full 30×30
diag_tjpcov = np.diag(S2.covariance.covmat)

# Retrieve ell centres and data-type labels
ell_gg,   cl_gg   = S2.get_ell_cl("galaxy_density_cl",        "lens0", "lens0")
ell_ee,   cl_ee   = S2.get_ell_cl("galaxy_shear_cl_ee",       "src0",  "src0")
ell_ge,   cl_ge   = S2.get_ell_cl("galaxy_shearDensity_cl_e", "lens0", "src0")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = [r"$C_\ell^{gg}$", r"$C_\ell^{\gamma\gamma}$", r"$C_\ell^{g\gamma}$"]
for ax, ell, cl, title in zip(axes,
                               [ell_gg, ell_ee, ell_ge],
                               [cl_gg,  cl_ee,  cl_ge],
                               titles):
    ax.loglog(ell, cl, "k-", label="fiducial $C_\\ell$")
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell$")
    ax.set_title(title)
    ax.legend()

plt.suptitle("1-bin fiducial 3×2pt data vector (TJPCov covariance)", fontsize=13)
plt.tight_layout()
plt.show()


### 7.4 Visualise the covariance matrix

The 30×30 TJPCov covariance has a clear block structure: the diagonal
blocks are the auto-covariances of each probe pair, and the off-diagonal
blocks are the cross-covariances.


In [ ]:
cov = S2.covariance.covmat
n   = cov.shape[0]          # 30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: correlation matrix (normalised) ---
std = np.sqrt(np.diag(cov))
corr = cov / np.outer(std, std)
im0 = axes[0].imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto",
                     extent=[-0.5, n - 0.5, n - 0.5, -0.5])
plt.colorbar(im0, ax=axes[0], label="Correlation")
axes[0].set_title("TJPCov correlation matrix")

# Mark block boundaries at ell bins per probe
for b in [10, 20]:
    axes[0].axhline(b - 0.5, color="k", linewidth=0.8)
    axes[0].axvline(b - 0.5, color="k", linewidth=0.8)

axes[0].set_xticks([5, 15, 25]); axes[0].set_xticklabels(
    [r"$gg$", r"$\gamma\gamma$", r"$g\gamma$"])
axes[0].set_yticks([5, 15, 25]); axes[0].set_yticklabels(
    [r"$gg$", r"$\gamma\gamma$", r"$g\gamma$"])

# --- Right: log|cov| ---
im1 = axes[1].imshow(np.log10(np.abs(cov)), cmap="viridis", aspect="auto",
                     extent=[-0.5, n - 0.5, n - 0.5, -0.5])
plt.colorbar(im1, ax=axes[1], label=r"$\log_{10}|C_{ij}|$")
axes[1].set_title(r"TJPCov $\log_{10}|$covariance$|$")
for b in [10, 20]:
    axes[1].axhline(b - 0.5, color="w", linewidth=0.8)
    axes[1].axvline(b - 0.5, color="w", linewidth=0.8)

plt.suptitle(r"1-bin 3×2pt covariance  ($f_\mathrm{sky}=0.3$, TJPCov FourierGaussianFsky)",
             fontsize=12)
plt.tight_layout()
plt.show()

print(f"Condition number (log10): {np.log10(np.linalg.cond(cov)):.1f}")


---
## 8. CLI Usage

All of the above can also be run in a single command:

```bash
augur srd_y1_3x2.yml            # GR forecast (run from tutorial dir)
augur 1bin_3x2pt_tjpcov.yml    # 1-bin with TJPCov covariance
augur -v srd_y1_3x2.yml        # verbose mode
```

The `-v` flag enables debug logging, showing each derivative step.

---

## Summary

| Step | Python API | Purpose |
|------|-----------|---------|
| Parse config | `augur.parse(config)` | Resolve Jinja2 templates + numpy expressions |
| Generate data vector | `generate(config, return_all_outputs=True)` | Build SACC file at fiducial cosmology |
| Fisher matrix | `Analyze(config, ...).get_fisher_matrix()` | Numerical derivatives → $F_{ij}$ |
| Systematic bias | `analysis.get_fisher_bias()` | Propagate parameter shifts → $\delta\theta$ |
| Postprocess | `postprocess(config)` | Triangle plots, pair plots, FoM table |

For a full description of all config options, see [config_guide.md](config_guide.md).


---
## 9. Configuration Examples

This section collects YAML snippets illustrating common configuration
patterns. Each snippet is a *partial* config showing only the relevant
keys — drop them into your own config alongside the required sections
(`cosmo`, `sources`, `lenses`, `statistics`, etc.).


### 9.1 Scale Cuts

Two approaches are available for restricting the angular scales included
in the data vector:

| Key | Effect |
|-----|--------|
| `kmax` (Mpc⁻¹) | Per-statistic physical scale cut; converted to $\ell_{\max}$ via $\ell_{\max} = k_{\max}\,\chi(\bar{z})$ |
| `lmax` | Direct multipole cut (scalar or per-tracer list) |
| `ignore_scale_cuts: False` | Enforce cuts when building the data vector (default behavior) |

```yaml
general:
    ignore_scale_cuts: False          # enforce kmax / lmax

statistics:
    galaxy_density_cl:
        tracer_combs: [[0, 0], [1, 1], [2, 2], [3, 3], [4, 4]]
        ell_edges: np.geomspace(20, 15000, 21, endpoint=True)
        kmax: 0.2          # physical cut in Mpc⁻¹ (conservative)
    galaxy_shear_cl_ee:
        tracer_combs: [[0, 0], ...]
        ell_edges: np.geomspace(20, 15000, 21, endpoint=True)
        kmax: null         # no cut for cosmic shear
    galaxy_shearDensity_cl_e:
        tracer_combs: [[0, 2], ...]
        ell_edges: np.geomspace(20, 15000, 21, endpoint=True)
        lmax: 2000         # direct ell_max instead of kmax
```

> **Tip**: `ignore_scale_cuts: True` includes all multipoles up to the
> `ell_edges` maximum regardless of `kmax`/`lmax`. Useful for quick tests.
> Set it to `False` for a realistic forecast.


### 9.2 Alternative Systematic Models

#### TATT intrinsic alignment model

Replace `LinearAlignmentSystematicFactory` with `TattAlignmentSystematicFactory`
in the `weak_lensing_factories` block:

```yaml
Firecrown_Factory:
    TwoPointFactory:
        weak_lensing_factories:
            - type_source: default
              global_systematics:
                - type: TattAlignmentSystematicFactory
                  include_z_dependence: True
              per_bin_systematics:
                - {type: MultiplicativeShearBiasFactory}
                - {type: PhotoZShiftFactory}
```

#### Redshift-space distortions in number counts

Set `include_rsd: true` in the `number_counts_factories` block:

```yaml
Firecrown_Factory:
    TwoPointFactory:
        number_counts_factories:
            - type_source: default
              include_rsd: true     # adds RSD contribution to C_ell^gg
              global_systematics: []
              per_bin_systematics:
                - {type: PhotoZShiftandStretchFactory}
                - {type: LinearBiasSystematicFactory}
```

#### Removing IA or multiplicative bias from the forecast

Simply omit those factories from `global_systematics` / `per_bin_systematics`.
No IA at all:

```yaml
        weak_lensing_factories:
            - type_source: default
              global_systematics: []       # no IA
              per_bin_systematics:
                - {type: PhotoZShiftFactory}
```


### 9.3 Non-Limber Integration (FKEM)

By default (`int_options: null`) all $C_\ell$ integrals use the Limber
approximation, which is accurate at $\ell \gtrsim 100$ but underestimates
power at large angular scales. The **FKEM** algorithm computes the exact
spherical Bessel integral at low $\ell$, then switches to Limber above a
threshold.

Two FKEM modes are available:

| `method` | Behavior |
|----------|----------|
| `fkem_auto` | Switch to Limber automatically when the relative Limber error is below `limber_max_error` |
| `fkem_l_limber` | Switch to Limber above a fixed `l_limber` |

```yaml
Firecrown_Factory:
    TwoPointFactory:
        # --- automatic switch-over based on accuracy target ---
        int_options:
            method: fkem_auto
            limber_method: gsl_qag_quad
            limber_max_error: 0.005   # switch to Limber when Limber error < 0.5%

        # --- OR: fixed switch-over multipole ---
        # int_options:
        #     method: fkem_l_limber
        #     limber_method: gsl_qag_quad
        #     l_limber: 300           # use full integral for ell < 300
```

> **Note**: Non-Limber integration significantly increases runtime
> (typically 10–50×). Use `fkem_l_limber` with `l_limber: 100–300`
> as a practical compromise for 3×2pt forecasts.


### 9.4 Gaussian Priors

Add a `gaussian_priors` block under `fisher` to impose Gaussian external
priors on nuisance parameters. The value is the prior $1\sigma$ width;
augur adds $1/\sigma^2$ to the Fisher diagonal.

```yaml
fisher:
    var_pars: ['Omega_c', 'sigma8', 'w0', 'wa', 'Omega_b', 'h', 'n_s',
               'lens0_bias', 'lens1_bias',
               'src0_delta_z', 'src1_delta_z',
               'lens0_delta_z', 'lens1_delta_z',
               'src0_mult_bias', 'src1_mult_bias']
    step: 1e-2
    gaussian_priors:
        # photo-z calibration — per-bin shift uncertainty
        src0_delta_z:  0.002    # 2 milli-dex, typical LSST Y1 source
        src1_delta_z:  0.002
        lens0_delta_z: 0.005    # 5 milli-dex, typical LSST Y1 lens
        lens1_delta_z: 0.005
        # shear calibration
        src0_mult_bias: 0.013   # 1.3% multiplicative bias prior
        src1_mult_bias: 0.013
        # galaxy bias (self-calibration)
        lens0_bias: 0.9
        lens1_bias: 0.9
```

> **Tip**: Even if a nuisance parameter is included in `var_pars` but has
> a tight external prior, its effect on the cosmological constraints will be
> correctly marginalised. You can also add priors to parameters *not* in
> `var_pars` — they are silently ignored.


### 9.5 Explicit Parameter Bounds and Normalized Stepping

The default `var_pars` mode steps around the fiducial cosmology with a fixed
fractional step `step`. For parameters with very different dynamic ranges,
normalised stepping — where the step is expressed as a fraction of the prior
range — can improve numerical accuracy.

Switch to **`parameters`** mode (explicit bounds) and pass `norm_step=True`
to `Analyze`:

```yaml
fisher:
    # Mode B: explicit bounds [min, fiducial, max]
    parameters:
        Omega_c: [0.06,  0.2664, 0.46]
        sigma8:  [0.3,   0.831,  1.2]
        n_s:     [0.87,  0.9645, 1.06]
        w0:      [-3.0, -1.0,   -0.3]
        wa:      [-3.0,  0.0,    3.0]
        Omega_b: [0.03,  0.0492, 0.07]
        h:       [0.55,  0.6727, 0.85]
        lens0_bias: [0.5, 1.562, 3.0]
    step: 0.01   # 1% of the parameter range [max - min]
    output: output/fisher.dat
    fid_output: output/fiducials.dat
```

In Python, pass `norm_step=True` to `Analyze`:

```python
analysis_normed = Analyze(
    parsed_config,            # config with `parameters` block above
    likelihood=likelihood,
    tools=tools,
    req_params=req_params,
    norm_step=True,           # step = 0.01 × (max - min) per parameter
)
analysis_normed.get_fisher_matrix()
```

> `norm_step` is only effective when `parameters` (with bounds) is used.
> With `var_pars`, the step is always the raw value of `step`.


### 9.6 Parameter Transforms: $\Omega_m$ and $S_8$

Add two boolean flags to the `fisher:` section to rotate the Fisher matrix
from the native $(\Omega_c, \sigma_8)$ basis into the more physically
intuitive $(\Omega_m, S_8)$ basis via the Jacobian transformation:

$$\Omega_m = \Omega_c + \Omega_b \quad (+ \Omega_\nu)$$
$$S_8 = \sigma_8 \sqrt{\Omega_m / 0.3}$$

```yaml
fisher:
    var_pars: ['Omega_c', 'sigma8', 'w0', 'wa', 'Omega_b', 'h', 'n_s',
               'lens0_bias', 'lens1_bias']
    transform_Omega_m: True   # Omega_c → Omega_m
    transform_S8: True        # sigma8  → S8 = sigma8 * sqrt(Omega_m / 0.3)
    ...
```

The cell below demonstrates the effect live using the Fisher matrix already
computed in Section 4. Note that `srd_y1_3x2.yml` already has these flags
enabled — the `postprocess` pairplot in Section 6 therefore shows $\Omega_m$
and $S_8$ contours.


In [ ]:
# Re-run Analyze with parameter transforms enabled
# (srd_y1_3x2.yml already has transform_Omega_m/S8 = True;
#  here we show the effect explicitly for illustration)
config_t = deepcopy(parsed_config)
config_t["fisher"]["transform_Omega_m"] = True
config_t["fisher"]["transform_S8"]      = True

analysis_t = Analyze(
    config_t,
    likelihood=likelihood,
    tools=tools,
    req_params=req_params,
)
analysis_t.get_fisher_matrix()

# Original vs transformed parameter names
print("Original basis :", var_pars)
print("Transformed basis:", analysis_t.var_pars)

# Marginalised 1-sigma in transformed basis
inv_t = np.linalg.inv(analysis_t.Fij)
sigs_t = np.sqrt(np.diag(inv_t))
sigs_orig = np.sqrt(np.diag(np.linalg.inv(fisher)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pars, sigs, title in [
    (axes[0], var_pars,           sigs_orig, r"Original: $(\Omega_c, \sigma_8)$"),
    (axes[1], analysis_t.var_pars, sigs_t,    r"Transformed: $(\Omega_m, S_8)$"),
]:
    ax.barh(pars, sigs, color="steelblue")
    ax.set_xlabel(r"Marginalised $1\sigma$")
    ax.set_title(title)
    for i, (p, s) in enumerate(zip(pars, sigs)):
        ax.text(s * 1.02, i, f"{s:.4f}", va="center", fontsize=8)

plt.suptitle("Effect of $(\\Omega_c,\\sigma_8)\\to(\\Omega_m,S_8)$ transform",
             fontsize=12)
plt.tight_layout()
plt.show()
